**Generative AI Use**: For the purposes of the assignments, the use of generative AI is subject to the same policies regarding collaboration. Just as with other collaborators, each student must write down the solutions independently of the output of the interaction and the submission should include a note denoting the nature of the collaboration. The use of generative AI tools to substantially complete sections of the assignments is not in line with the spirit of the assignments, and would be a violation of the [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code).

---

**中文对照（参考译文，不含解答）**

**生成式 AI 的使用**：本作业对生成式 AI 的使用采用与协作相同的规则。与其他协作者交流之后，每位学生仍须独立写出自己的解答，不能直接依赖交流中得到的输出；提交时应附上说明，如实记录协作的性质。使用生成式 AI 实质性地完成作业的大部分内容，不符合本作业的教学目的，也会违反[学术诚信准则](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

**Student Declaration (must be completed)**

**SUNet id:**  
_Your SUNet id_

**Did you use generative AI tools (e.g., ChatGPT, Copilot, etc.)?**  
- [ ] No  
- [ ] Yes (please describe below)

**If yes, describe how you used generative AI and for what parts of the assignment :**  
_Your Answer_

**Confirm that all submitted work reflects your own understanding and that you did not rely on AI to complete substantial portions of the assignment:**  
- [ ] I confirm

---

**中文对照（参考译文，不含解答）**

**学生声明（必须填写）**

**SUNet 账号：** [请自行填写]

**你是否使用了生成式 AI 工具（例如 ChatGPT、Copilot 等）？**
- [ ] 否
- [ ] 是（请在下方说明）

**如果使用过，请说明使用方式，以及涉及作业的哪些部分：** [请自行填写]

**请确认：提交的内容反映了你自己的理解，并且你没有依靠 AI 完成作业的实质性部分。**
- [ ] 我确认

In [ ]:
# This mounts your Google Drive to the Colab VM.
# [中文] 这把Google车开到Colab VM
from google.colab import drive
drive.mount('/content/drive')

# TODO: Enter the foldername in your Drive where you have saved the unzipped
# assignment folder, e.g. 'cs231n/assignments/assignment3/'
# [中文] TODO: 在您的驱动器中输入文件夹名, 在那里您保存了未覆盖的任务文件夹e.g。 “ cs231n/ assignations/
# formantions3/ ”
FOLDERNAME = 'cs231n/assignments/assignment3/'
assert FOLDERNAME is not None, "[!] Enter the foldername."

# Now that we've mounted your Drive, this ensures that
# the Python interpreter of the Colab VM can load
# python files from within it.
# [中文] 现在,我们已经安装了你的驱动器, 这确保了Python 翻译 科拉布VM 可以装载
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# This downloads the COCO dataset to your Drive
# if it doesn't already exist.
# [中文] 如果COCO数据不存在, 它会下载到驱动器上。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
# !bash get_datasets.sh
!bash get_coco_dataset.sh
%cd /content/drive/My\ Drive/$FOLDERNAME


# Image Captioning with Transformers
You have now implemented a vanilla RNN and for the task of image captioning. In this notebook you will implement key pieces of a transformer decoder to accomplish the same task.

**NOTE:** This notebook will be primarily written in PyTorch rather than NumPy, unlike the RNN notebook.

---

**中文对照（参考译文，不含解答）**

# 带有变形器的图像说明
您已经实施了 Vanilla RNN 和图像字幕任务。 在此笔记本中您将执行Transformer解码器的关键部件来完成同样的任务。

** 这本笔记本将主要用PyTorrch而不是NumPy撰写,与RNN笔记本不同。

In [ ]:
# Setup cell.
# [中文] 设置牢房。
import time, os, json
import numpy as np
import matplotlib.pyplot as plt
import sys
import types
import importlib

from cs231n.gradient_check import eval_numerical_gradient, eval_numerical_gradient_array
from cs231n.transformer_layers import *
from cs231n.captioning_solver_transformer import CaptioningSolverTransformer
from cs231n.classifiers.transformer import CaptioningTransformer
from cs231n.coco_utils import load_coco_data, sample_coco_minibatch, decode_captions
from cs231n.image_utils import image_from_url

%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # Set default size of plots.
# [中文] 设置默认的绘图大小。
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

if "imp" not in sys.modules:
    imp = types.ModuleType("imp")
    imp.reload = importlib.reload
    sys.modules["imp"] = imp

%load_ext autoreload
%autoreload 2

def rel_error(x, y):
    """ returns relative error """
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

# COCO Dataset
As in the previous notebooks, we will use the COCO dataset for captioning.

---

**中文对照（参考译文，不含解答）**

# COCO 数据集
与前几本笔记本一样,我们将使用COCO数据集进行字幕说明。

In [ ]:
# Load COCO data from disk into a dictionary.
# [中文] 将 COCO 数据从磁盘装入字典。
data = load_coco_data(pca_features=True)

# Print out all the keys and values from the data dictionary.
# [中文] 从数据字典中打印所有键和值。
for k, v in data.items():
    if type(v) == np.ndarray:
        print(k, type(v), v.shape, v.dtype)
    else:
        print(k, type(v), len(v))

# Transformer
As you have seen, RNNs are incredibly powerful but often slow to train. Further, RNNs struggle to encode long-range dependencies (though LSTMs are one way of mitigating the issue). In 2017, Vaswani et al introduced the Transformer in their paper ["Attention Is All You Need"](https://arxiv.org/abs/1706.03762) to a) introduce parallelism and b) allow models to learn long-range dependencies. The paper not only led to famous models like BERT and GPT in the natural language processing community, but also an explosion of interest across fields, including vision. While here we introduce the model in the context of image captioning, the idea of attention itself is much more general.



---

**中文对照（参考译文，不含解答）**

# Transformer
如您所见, RNNs 具有惊人的威力,但训练速度往往很慢。 此外, RNS 努力编码远程依赖关系(though LSTMs are one way of mitigating the issue)。 在2017中, Vaswani 等人在他们的论文["Attention Is All You Need"](https://arxiv.org/abs/1706.03762) 中将变形器引入了 a) 引入了平行主义, (b) 允许模型学习远程依赖关系。 该论文不仅在自然语言处理界产生了著名的模型, 而且还引起了包括视觉在内的各个领域的极大兴趣。 虽然我们在这里在图像说明中引入了模型, 关注的概念本身则更为笼统。



# Transformer: Multi-Headed Attention

### Dot-Product Attention

Recall that attention can be viewed as an operation on a query $q\in\mathbb{R}^d$, a set of value vectors $\{v_1,\dots,v_n\}, v_i\in\mathbb{R}^d$, and a set of key vectors $\{k_1,\dots,k_n\}, k_i \in \mathbb{R}^d$, specified as

---

**中文对照（参考译文，不含解答）**

# 变换器: 多头关注

### 点- 过程注意

提醒注意可视为查询 $q\in\mathbb{R}^d$ 、一组值矢量 $\{v_1,\dots,v_n\}, v_i\in\mathbb{R}^d$ 和一组键矢量 $\{k_1,\dots,k_n\}, k_i \in \mathbb{R}^d$ 上的操作。

\begin{align}
c &= \sum_{i=1}^{n} v_i \alpha_i \\
\alpha_i &= \frac{\exp(k_i^\top q)}{\sum_{j=1}^{n} \exp(k_j^\top q)} \\
\end{align}

where $\alpha_i$ are frequently called the "attention weights", and the output $c\in\mathbb{R}^d$ is a correspondingly weighted average over the value vectors.

### Self-Attention
In Transformers, we perform self-attention, which means that the values, keys and query are derived from the input $X \in \mathbb{R}^{\ell \times d}$, where $\ell$ is our sequence length. Specifically, we learn parameter matrices $V,K,Q \in \mathbb{R}^{d\times d}$ to map our input $X$ as follows:

---

**中文对照（参考译文，不含解答）**

点$\alpha_i$通常被称为“注意权重”和输出$c\in\mathbb{R}^d$相对于值矢量的相应加权平均值。

### 自注意力
在变换器中,我们进行自我注意,这意味着值、键和查询来自输入$X \in \mathbb{R}^{\ell \times d}$,其中 $\ell$是我们的序列长度。具体地说,我们学习参数矩阵$V,K,Q \in \mathbb{R}^{d\times d}$来绘制输入$X$的图解如下:

\begin{align}
v_i = Vx_i\ \ i \in \{1,\dots,\ell\}\\
k_i = Kx_i\ \ i \in \{1,\dots,\ell\}\\
q_i = Qx_i\ \ i \in \{1,\dots,\ell\}
\end{align}

### Multi-Headed Scaled Dot-Product Attention
In the case of multi-headed attention, we learn a parameter matrix for each head, which gives the model more expressivity to attend to different parts of the input. Let $h$ be number of heads, and $Y_i$ be the attention output of head $i$. Thus we learn individual matrices $Q_i$, $K_i$ and $V_i$. To keep our overall computation the same as the single-headed case, we choose $Q_i \in \mathbb{R}^{d\times d/h}$, $K_i \in \mathbb{R}^{d\times d/h}$ and $V_i \in \mathbb{R}^{d\times d/h}$. Adding in a scaling term $\frac{1}{\sqrt{d/h}}$ to our simple dot-product attention above, we have

---

**中文对照（参考译文，不含解答）**

### 多头缩放点- Production 注意
在多头关注的情况下, 我们为每个头部学习一个参数矩阵, 使模型能更直观地处理输入的不同部分。 让 $h$ 成为头数, 并且 $Y_i$ 成为标题 $i$ 的注意输出。 因此, 我们学习单个矩阵 $Q_i$ 、 $K_i$ 和 $V_i$。 为了保持与单头案例相同的整体计算, 我们选择了 $Q_i \in \mathbb{R}^{d\times d/h}$ 、$K_i \in \mathbb{R}^{d\times d/h}$ 和$V_i \in \mathbb{R}^{d\times d/h}$。 在一个缩放术语 $\frac{1}{\sqrt{d/h}}$ 中添加到我们上面简单的点产品关注上方。

\begin{equation} \label{qkv_eqn}
Y_i = \text{softmax}\bigg(\frac{(XQ_i)(XK_i)^\top}{\sqrt{d/h}}\bigg)(XV_i)
\end{equation}

where $Y_i\in\mathbb{R}^{\ell \times d/h}$, where $\ell$ is our sequence length.

In our implementation, we apply dropout to the attention weights (though in practice it could be used at any step):

---

**中文对照（参考译文，不含解答）**

其中 $Y_i\in\mathbb{R}^{\ell \times d/h}$,其中$\ell$是我们的序列长度。

在我们的执行中,我们应用 Dropout（随机失活） 来表示对 (though in practice it could be used at any step) 的注意:

\begin{equation} \label{qkvdropout_eqn}
Y_i = \text{dropout}\bigg(\text{softmax}\bigg(\frac{(XQ_i)(XK_i)^\top}{\sqrt{d/h}}\bigg)\bigg)(XV_i)
\end{equation}

Finally, then the output of the self-attention is a linear transformation of the concatenation of the heads:

---

**中文对照（参考译文，不含解答）**

最后,自我关注的输出是头部的直线转换:

\begin{equation}
Y = [Y_1;\dots;Y_h]A
\end{equation}

were $A \in\mathbb{R}^{d\times d}$ and $[Y_1;\dots;Y_h]\in\mathbb{R}^{\ell \times d}$.

Implement multi-headed scaled dot-product attention in the `MultiHeadAttention` class in the file `cs231n/transformer_layers.py`. The code below will check your implementation. The relative error should be less than `7e-3`.

---

**中文对照（参考译文，不含解答）**

是 $A \in\mathbb{R}^{d\times d}$ 和 $[Y_1;\dots;Y_h]\in\mathbb{R}^{\ell \times d}$。

实施多头裁量的点产品关注`MultiHeadAttention`文件的类`cs231n/transformer_layers.py`。以下代码将检查您的执行。相对错误应小于`7e-3` .

In [ ]:
torch.manual_seed(231)

# Choose dimensions such that they are all unique for easier debugging:
# Specifically, the following values correspond to N=1, H=2, T=3, E//H=4, and E=8.
# [中文] 选择便于调试的尺寸:具体来说,以下值对应N=1、H=2、T=3、E/H= 4和E=8。
batch_size = 1
sequence_length = 3
embed_dim = 8
attn = MultiHeadAttention(embed_dim, num_heads=2)
attn.eval()

# Self-attention.
# [中文] 自我关注。
data = torch.randn(batch_size, sequence_length, embed_dim)
self_attn_output = attn(query=data, key=data, value=data)

# Masked self-attention.
# [中文] 蒙面的自我关注。
mask = torch.randn(sequence_length, sequence_length) < 0.5
masked_self_attn_output = attn(query=data, key=data, value=data, attn_mask=mask)

# Attention using two inputs.
# [中文] 注意使用两个输入。
other_data = torch.randn(batch_size, sequence_length, embed_dim)
attn_output = attn(query=data, key=other_data, value=other_data)

expected_self_attn_output = np.asarray([[
    [-0.1030,  0.2090,  0.7481, -0.4116, -0.2772,  0.2357, -0.2740, -0.2798],
    [-0.1794,  0.1637,  0.6930, -0.3435, -0.2300,  0.2457, -0.2504, -0.2323],
    [-0.1861,  0.1567,  0.6677, -0.3078, -0.2731,  0.2302, -0.2714, -0.2082]]])

expected_masked_self_attn_output = np.asarray([[
    [-0.1209,  0.1806,  0.8056, -0.4654, -0.2262,  0.2552, -0.2544, -0.2812],
    [-0.2374,  0.1183,  0.7423, -0.3777, -0.1460,  0.2716, -0.2213, -0.2077],
    [-0.0851,  0.2423,  0.7941, -0.3778, -0.4924,  0.1512, -0.4241, -0.1740]]])

expected_attn_output = np.asarray([[
    [-0.1965,  0.0702,  0.5227, -0.1388, -0.3530,  0.2163, -0.2643, -0.1902],
    [-0.3592, -0.0899,  0.5836,  0.0029, -0.3643,  0.2667, -0.1973, -0.1698],
    [-0.3795,  0.0433,  0.5634, -0.0749, -0.4379,  0.2714, -0.2207, -0.1980]]])

print('self_attn_output error: ', rel_error(expected_self_attn_output, self_attn_output.detach().numpy()))
print('masked_self_attn_output error: ', rel_error(expected_masked_self_attn_output, masked_self_attn_output.detach().numpy()))
print('attn_output error: ', rel_error(expected_attn_output, attn_output.detach().numpy()))

# Positional Encoding

While transformers are able to easily attend to any part of their input, the attention mechanism has no concept of token order. However, for many tasks (especially natural language processing), relative token order is very important. To recover this, the authors add a positional encoding to the embeddings of individual word tokens.

Let us define a matrix $P \in \mathbb{R}^{l\times d}$, where $P_{ij} = $

---

**中文对照（参考译文，不含解答）**

# 位置编码

虽然Transformer能够轻松处理其输入的任何部分, 但注意机制没有象征性命令的概念。 但是, 对于许多任务 (especially natural language processing), 相对的象征性命令非常重要。 要恢复这一点, 作者会在单个单词标记的嵌入中添加位置编码。

让我们定义一个 $P \in \mathbb{R}^{l\times d}$ 的矩阵, 其中的 $P_{ij} = $

$$
\begin{cases}
\text{sin}\left(i \cdot 10000^{-\frac{j}{d}}\right) & \text{if j is even} \\
\text{cos}\left(i \cdot 10000^{-\frac{(j-1)}{d}}\right) & \text{otherwise} \\
\end{cases}
$$

Rather than directly passing an input $X \in \mathbb{R}^{l\times d}$ to our network, we instead pass $X + P$.

Implement this layer in `PositionalEncoding` in `cs231n/transformer_layers.py`. Once you are done, run the following to perform a simple test of your implementation. You should see errors on the order of `e-3` or less.

---

**中文对照（参考译文，不含解答）**

我们不是直接将输入$X \in \mathbb{R}^{l\times d}$传送到我们的网络,而是通过$X + P$。

执行此层`PositionalEncoding`在`cs231n/transformer_layers.py`。一旦完成后,请运行以下操作来对您的执行进行简单的测试。您应该看到以下顺序上的错误:`e-3`或更少。 或少于。。 \fn微软雅黑Or less. \fs18\b1\bord1\shad1\3cH2F2F3F}。

In [ ]:
torch.manual_seed(231)

batch_size = 1
sequence_length = 2
embed_dim = 6
data = torch.randn(batch_size, sequence_length, embed_dim)

pos_encoder = PositionalEncoding(embed_dim)
pos_encoder.eval()
output = pos_encoder(data)

expected_pe_output = np.asarray([[[-1.1106,  1.0014,  1.5280, -0.0778, -0.6964,  1.1455],
                                  [ 0.8125, -0.4303,  0.4981,  0.7320,  1.1380,  1.5331]]])
print('pe_output error: ', rel_error(expected_pe_output, output.detach().numpy()))

# Inline Question 1

Several key design decisions were made in designing the scaled dot product attention we introduced above. Explain why the following choices were beneficial:
1. Using multiple attention heads as opposed to one.
2. Dividing by $\sqrt{d/h}$ before applying the softmax function. Recall that $d$ is the feature dimension and $h$ is the number of heads.
3. Adding a linear transformation to the output of the attention operation.

Only one or two sentences per choice is necessary, but be sure to be specific in addressing what would have happened without each given implementation detail, why such a situation would be suboptimal, and how the proposed implementation improves the situation.

**Your Answer:**


---

**中文对照（参考译文，不含解答）**

# 随堂问题 1

上文介绍的缩放点积注意力包含几项关键设计。请解释以下选择为什么有益：

1. 使用多个注意力头。
2. 在应用 Softmax 函数之前除以 $\sqrt{d/h}$。其中，$d$ 是特征维度，$h$ 是注意力头的数量。
3. 对注意力运算的输出添加一个线性变换。

每项用一两句话说明即可，但需要具体指出：如果缺少这一设计会发生什么，为什么那种情况不理想，以及该设计如何改善这种情况。

**你的回答：** [请自行填写]

# Transformer Decoder Block

Transformer decoder layer consists of three modules: (1) self attention to process input sequence of vectors, (2) cross attention to process based on available context (i.e. image features in our case), (3) feedforward module to process each vector of the sequence independently. Complete the implementation of `TransformerDecoderLayer` in `cs231n/transformer_layers.py` and test it below. The relative error should be less than 1e-6.

The Transformer decoder layer has three main components: (1) a self-attention module that processes the input sequence of vectors, (2) a cross-attention module that incorporates additional context (e.g., image features in our case), and (3) a feedforward module that independently processes each vector in the sequence. Complete the implementation of `TransformerDecoderLayer` in `cs231n/transformer_layers.py` and test it below. The relative error should be less than 1e-6.

---

**中文对照（参考译文，不含解答）**

# 变换解码器块

变换器解码器层由三个模块组成: (1) 自我关注矢量的流程输入序列, (2) 交叉关注基于可用上下文( i.e . 图像特性)的进程, (3) 向前反馈模块独立处理序列的每个矢量。完成 `TransformerDecoderLayer` 在 `cs231n/transformer_layers.py` 中的实施并测试它。相对错误应小于 1e-6。

变换器解码器层有三个主要组成部分:(1)处理矢量输入序列的自控模块,(2)包含额外上下文的交叉注意模块( )e.g...,图像特征 在我们的情况),和(3)在序列中独立处理每个矢量的反馈向前模块。`TransformerDecoderLayer`在`cs231n/transformer_layers.py`并在下面测试。 相对错误应小于1e-6 .

In [ ]:
torch.manual_seed(231)
np.random.seed(231)

N, T, TM, D = 1, 4, 5, 12

decoder_layer = TransformerDecoderLayer(D, 2, 4*D)
decoder_layer.eval()
tgt = torch.randn(N, T, D)
memory = torch.randn(N, TM, D)
tgt_mask = torch.randn(T, T) < 0.5

output = decoder_layer(tgt, memory, tgt_mask)

expected_output = np.asarray([
    [[ 0.27005595, -0.21056601,  0.63376445, -0.35391954,  0.60071295,
      -1.7566615,  -0.1314159,  -1.7659538,  -0.5575517,   0.680143,
       1.8622686,   0.7291234],
     [-0.6080135,   0.4421802,  -0.07590749, -0.12093157,  0.94722354,
      -1.0223433,   0.47332686, -1.3958666,   2.1890075,  -1.0380646,
       0.9437816,  -0.7343922],
     [-0.89690155,  0.60447216,  0.10688882,  0.03492759,  1.8388084,
      -0.9772293,   0.7621334,  -0.668234,    1.3260399,  -1.8417019,
       0.21718435, -0.506388],
     [-0.70961505, -0.02918372, -0.39894593, -0.8069395,   0.04967685,
       0.29345992,  0.7871747,  -1.3172938,   1.4744806,   1.2633642,
       1.1611108,  -1.767289]]
])
print('error: ', rel_error(expected_output, output.detach().numpy()))

# Transformer for Image Captioning
Now that you have implemented the previous layers, you can combine them to build a Transformer-based image captioning model. Open the file `cs231n/classifiers/transformer.py` and look at the `CaptioningTransformer` class.

Implement the `forward` function of the class. After doing so, run the following to check your forward pass using a small test case; you should see error on the order of `e-5` or less.

---

**中文对照（参考译文，不含解答）**

# 图像说明的变换器
既然您已经实施了前几层, 您可以将其合并来构建一个基于变换器的图像字幕模型。 打开文件 `cs231n/classifiers/transformer.py` 并查看 `CaptioningTransformer` 类。

执行该类的 `forward` 函数。 这样做后, 运行以下功能, 使用小测试框检查您的 前向传播; 您应该看到 `e-5` 或更少的顺序上的错误。

In [ ]:
torch.manual_seed(231)
np.random.seed(231)

N, D, W = 4, 20, 30
word_to_idx = {'<NULL>': 0, 'cat': 2, 'dog': 3}
V = len(word_to_idx)
T = 3

transformer = CaptioningTransformer(
    word_to_idx,
    input_dim=D,
    wordvec_dim=W,
    num_heads=2,
    num_layers=2,
    max_length=30
)
transformer.eval()

features = torch.randn(N, D)
captions = torch.randint(0, V, (N, T))

scores = transformer(features, captions)

expected_scores = np.asarray([
    [[ 0.62165695, -0.6203744,  -1.116114  ],
     [ 0.6641097,  -0.60309553, -1.3479083 ],
     [ 0.62117404, -0.79205745, -1.1095243 ]],
    [[ 0.62160987, -0.6214724,  -1.1171186 ],
     [ 0.6093992,  -0.61166143, -1.3365762 ],
     [ 0.62108713, -0.79322916, -1.1104958 ]],
    [[ 0.6406587,  -0.57419956, -1.1300566 ],
     [ 0.6075228,  -0.60872054, -1.3372457 ],
     [ 0.6471472,  -0.8270494,  -1.1064028 ]],
    [[ 0.64029807, -0.5755552,  -1.1284486 ],
     [ 0.60708034, -0.6099963,  -1.3356377 ],
     [ 0.59367496, -0.8346836,  -1.0914496 ]]
])

print('scores error: ', rel_error(expected_scores, scores.detach().numpy()))

# Overfit Transformer Captioning Model on Small Data
Run the following to overfit the Transformer-based captioning model on the same small dataset as we used for the RNN previously.

---

**中文对照（参考译文，不含解答）**

# 小数据超配变形说明模型
运行以下数据, 以将基于变换器的字幕模型与以前用于 RNN 的同一小数据集相匹配。

In [ ]:
torch.manual_seed(231)
np.random.seed(231)

data = load_coco_data(max_train=50)

transformer = CaptioningTransformer(
          word_to_idx=data['word_to_idx'],
          input_dim=data['train_features'].shape[1],
          wordvec_dim=256,
          num_heads=2,
          num_layers=2,
          max_length=30
        )


transformer_solver = CaptioningSolverTransformer(transformer, data, idx_to_word=data['idx_to_word'],
           num_epochs=100,
           batch_size=25,
           learning_rate=0.001,
           verbose=True, print_every=10,
         )

transformer_solver.train()

# Plot the training losses.
# [中文] 计算训练损失。
plt.plot(transformer_solver.loss_history)
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.title('Training loss history')
plt.show()

Print final training loss. You should see a final loss of less than 0.05 .

---

**中文对照（参考译文，不含解答）**

打印最后训练损失。 您应该看到最后损失小于 0.05。

In [ ]:
print('Final loss: ', transformer_solver.loss_history[-1])

# Transformer Sampling at Test Time
The sampling code has been written for you. You can simply run the following to compare with the previous results with the RNN. As before the training results should be much better than the validation set results, given how little data we trained on.

---

**中文对照（参考译文，不含解答）**

# 测试时的Transformer抽样
已经为您写入了抽样代码。 您可以简单地运行以下数据, 以便与先前的 RNN 比较。 训练结果应该比校验结果要好得多, 因为我们训练的数据很少。

In [ ]:
# If you get an error, the URL just no longer exists, so don't worry!
# You can re-sample as many times as you want.
# [中文] 如果您有错误, URL 已经不存在了, 所以不要担心 ! 您可以重试多少次。
for split in ['train', 'val']:
    minibatch = sample_coco_minibatch(data, split=split, batch_size=2)
    gt_captions, features, urls = minibatch
    gt_captions = decode_captions(gt_captions, data['idx_to_word'])

    sample_captions = transformer.sample(features, max_length=30)
    sample_captions = decode_captions(sample_captions, data['idx_to_word'])

    for gt_caption, sample_caption, url in zip(gt_captions, sample_captions, urls):
        img = image_from_url(url)
        # Skip missing URLs.
        # [中文] 跳过缺失的 URL。
        if img is None: continue
        plt.imshow(img)
        plt.title('%s\n%s\nGT:%s' % (split, sample_caption, gt_caption))
        plt.axis('off')
        plt.show()

# Vision Transformer (ViT)

[Dosovitskiy et. al.](https://arxiv.org/abs/2010.11929) showed that applying a transformer model on a sequence of image patches (referred to as Vision Transformer) not only achieves impressive performance but also scales more effectively than convolutional neural networks when trained on large datasets. We will build a version of Vision Transformer using our existing implementation of transformer components and train it on the CIFAR-10 dataset.

---

**中文对照（参考译文，不含解答）**

# 愿景转换器 (ViT)

 [Dosovitskiy et. al.](https://arxiv.org/abs/2010.11929)显示在图像补丁序列中应用Transformer模型(referred to as Vision Transformer)不仅取得了令人印象深刻的成绩,而且在接受大型数据集训练时,比卷积神经网络更为有效。 我们将利用现有的Transformer组件来构建一个愿景变异器版本,并对其进行关于CIFAR-CIFA的训练。10数据集。

Vision Transformer converts input image into a sequence of patches of fixed size and embed each patch into a latent vector. In `cs231n/transformer_layers.py`, complete the implementation of `PatchEmbedding` and test it below. You should see relative error less than 1e-4.

---

**中文对照（参考译文，不含解答）**

愿景转换器将输入图像转换成一个固定大小的图像块序列,并将每个图像块嵌入潜向矢量。在 `cs231n/transformer_layers.py` 中,完成 `PatchEmbedding` 的安装,并在下面测试。您应该看到低于 1e-4 的相对错误。

In [ ]:
from cs231n.transformer_layers import PatchEmbedding

torch.manual_seed(231)
np.random.seed(231)

N = 2
HW = 16
PS = 8
D = 8

patch_embedding = PatchEmbedding(
    img_size=HW,
    patch_size=PS,
    embed_dim=D
)

x = torch.randn(N, 3, HW, HW)
output = patch_embedding(x)

expected_output = np.asarray([
        [[-0.6312704 ,  0.02531429,  0.6112642 , -0.49089882,
          0.01412961, -0.6959372 , -0.32862484, -0.45402682],
        [ 0.18816411, -0.08142513, -0.9829535 , -0.23975623,
         -0.23109074,  0.97950286, -0.40997326,  0.7457837 ],
        [ 0.01810865,  0.15780598, -0.91804236,  0.36185235,
          0.8379501 ,  1.0191797 , -0.29667392,  0.20322265],
        [-0.18697818, -0.45137224, -0.40339014, -1.4381214 ,
         -0.43450755,  0.7651071 , -0.83683825, -0.16360264]],

       [[-0.39786366,  0.16201034, -0.19008337, -1.0602452 ,
         -0.28693503,  0.09791763,  0.26614824,  0.41781986],
        [ 0.35146567, -0.4469593 , -0.1841726 ,  0.45757473,
         -0.61304873, -0.29104248, -0.16124889, -0.14987172],
        [-0.2996967 ,  0.27353522, -0.09929767,  0.01973832,
         -1.2312065 , -0.6374332 , -0.22963578,  0.55696607],
        [-0.93818814,  0.02465284, -0.21117875,  1.1860403 ,
         -0.06137538, -0.21062079, -0.094347  ,  0.50032747]]])

print('error: ', rel_error(expected_output, output.detach().numpy()))

The sequence of patch vectors is processed by transformer encoder layers, each consisting of a self-attention and a feed-forward module. Since all vectors attend to one another, attention masking is not strictly necessary. However, we still implement it for the sake of consistency.

Implement `TransformerEncoderLayer` in `cs231n/transformer_layers.py` and test it below. You should see relative error less than 1e-6.


---

**中文对照（参考译文，不含解答）**

图像块矢量的序列由Transformer编码器层处理,每层由自我注意和进料传导模块组成。由于所有矢量都互相注意,因此严格来说,没有需要注意遮罩。然而,为了一致性,我们仍在执行它。

在 `cs231n/transformer_layers.py` 中执行 `TransformerEncoderLayer`,并在下面测试它。 您应该看到比 1e-6 少的相对错误。


In [ ]:
torch.manual_seed(231)
np.random.seed(231)

from cs231n.transformer_layers import TransformerEncoderLayer

N, T, TM, D = 1, 4, 5, 12

encoder_layer = TransformerEncoderLayer(D, 2, 4*D)
encoder_layer.eval()
x = torch.randn(N, T, D)
x_mask = torch.randn(T, T) < 0.5

output = encoder_layer(x, x_mask)

expected_output = np.asarray([
    [[-0.48887438, -0.16470742,  0.7249629,  -1.0907345,   1.5764195,
       0.17484017,  0.8643941,  -0.5209561,  -1.6651545,  -1.1943513,
       1.4809005,   0.30326116],
     [-0.14663944,  1.052812,   -0.96227,    -0.20350897, -2.1478589,
       1.2637341,   0.04015125, -0.945255,    1.408903,   -0.43051198,
       0.37110543,  0.69933903],
     [ 0.08718029,  0.91050833, -1.208469,    1.1440394,  -1.9794976,
      -0.38342738, -0.47407156, -0.17345548,  0.11738186,  1.9534125,
       0.3525986,  -0.3461999],
     [-0.81389004,  0.76213264, -0.7094313,  -1.5332246,   1.3837544,
       0.4091982,   0.6569406,   0.7305365,  -1.197077,   -1.3698943,
       0.58292854,  1.0980265]]
])

print('error: ', rel_error(expected_output, output.detach().numpy()))

Take a look at the `VisionTransformer` implementation in `cs231n/classifiers/transformer.py`.

For classification, ViT divides the input image into patches and processes the sequence of patch vectors using a transformer. Finally, all the patch vectors are average-pooled and used to predict the image class. We will use the same 1D sinusoidal positional encoding to inject ordering information, though 2D sinusoidal and learned positional encodings are also valid choices.

Complete the ViT forward pass and test it below. You should see relative error less than 1e-6.

---

**中文对照（参考译文，不含解答）**

查看 `VisionTransformer` 中的 `cs231n/classifiers/transformer.py` 执行。

对于分类, ViT 将输入图像分割为图像块, 并使用Transformer处理图像块矢量序列。 最后, 所有图像块矢值都是平均共享的, 并用来预测图像类。 我们将使用相同的 1D 类正弦位置编码来输入信息, 尽管 2D 类beta和学习的定位编码也是有效的选择。

完成 VT 前向传播 并在下面测试它。 您应该看到比 1e-6 少的相对错误。

In [ ]:
torch.manual_seed(231)
np.random.seed(231)
from cs231n.classifiers.transformer import VisionTransformer

imgs = torch.randn(3, 3, 32, 32)
transformer = VisionTransformer()
transformer.eval()

scores = transformer(imgs)

expected_scores = np.asarray(
    [[-0.15802915,  0.15734261, -0.06002094, -0.1860142,  -0.09376919,
      -0.08837911,  0.16224845,  0.0168601,   0.20606893,  0.13079852],
     [-0.12838936,  0.20798579, -0.0620573,  -0.11705838, -0.14088857,
      -0.11387511,  0.18839046,  0.03078492,  0.17386372,  0.06414902],
     [-0.09435399,  0.20495278, -0.03362986, -0.12232997, -0.15535004,
      -0.10380758,  0.21114832,  0.03481449,  0.18798208,  0.12231653]])

print('scores error: ', rel_error(expected_scores, scores.detach().numpy()))


We will first verify our implementation by overfitting it on one training batch. Tune learning rate and weight decay accordingly.

---

**中文对照（参考译文，不含解答）**


首先,我们将通过在一组训练中进行超额配置来核实我们的执行情况,从而降低学习率和体重。

In [ ]:
from torchvision import transforms
from torchvision.datasets import CIFAR10
from tqdm.auto import tqdm
from torch.utils.data import DataLoader

train_data = CIFAR10(root='data', train=True, transform=transforms.ToTensor(), download=True)
test_data = CIFAR10(root='data', train=False, transform=transforms.ToTensor(), download=True)

In [ ]:

learning_rate = 1e-4  # Experiment with this
# [中文] 试试这个
weight_decay = 1.e-4  # Experiment with this
# [中文] 试试这个


batch = next(iter(DataLoader(train_data, batch_size=64, shuffle=False)))
model = VisionTransformer(dropout=0.0)
loss_criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
model.train()

epochs = 100
for epoch in range(epochs):
    imgs, target = batch
    out = model(imgs)
    loss = loss_criterion(out, target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    top1 = (out.argmax(-1) == target).float().mean().item()
    if epoch % 10 == 0:
      print(f"[{epoch}/{epochs}] Loss {loss.item():.6f}, Top-1 Accuracy: {top1:.3f}")


In [ ]:
# You should get perfect 1.00 accuracy
# [中文] 您应该获得完美的 1.00 精确度
print(f"Overfitting ViT on one batch. Top-1 accuracy: {top1}")

Now we will train it on the entire dataset.

---

**中文对照（参考译文，不含解答）**

现在我们要用整个数据集来训练它。

In [ ]:
from cs231n.classification_solver_vit import ClassificationSolverViT

############################################################################
# TODO: Train a Vision Transformer model that achieves over 0.45 test      #
# accuracy on CIFAR-10 after 2 epochs by adjusting the model architecture  #
# and/or training parameters as needed.                                    #
#                                                                          #
# Note: If you want to use a GPU runtime, go to `Runtime > Change runtime  #
# type` and set `Hardware accelerator` to `GPU`. This will reset Colab,    #
# so make sure to rerun the entire notebook from the beginning afterward.  #
############################################################################
# [中文] TODO: 训练一个能够实现超额目标的愿景转换模型0.45CIFAR的测试准确率10之后2通过根据需要调整模型架构和/或训练参数来达到训练轮次。 注意:
# 如果您想要使用GPU运行时间,去`Runtime > Change runtime type`和设置`Hardware
# accelerator`到`GPU`。这将重置 Colab,以确保从头开始重新运行整个笔记本。


learning_rate = 1e-4
weight_decay = 0.0
batch_size = 64
model = VisionTransformer()  # You may want to change the default params.
# [中文] 您可能想要更改默认参数。




################################################################################
#                                 END OF YOUR CODE                             #
################################################################################
# [中文] 代码结束

solver = ClassificationSolverViT(
    train_data=train_data,
    test_data=test_data,
    model=model,
    num_epochs = 2,  # Don't change this
    # [中文] 不要改变这个
    learning_rate = learning_rate,
    weight_decay = weight_decay,
    batch_size = batch_size,
)

solver.train('cuda' if torch.cuda.is_available() else 'cpu')



In [ ]:
print(f"Accuracy on test set: {solver.results['best_test_acc']}")

# Inline Question 2

Despite their recent success in large-scale image recognition tasks, ViTs often lag behind traditional CNNs when trained on smaller datasets. What underlying factor contribute to this performance gap? What techniques can be used to improve the performance of ViTs on small datasets?

**Your Answer**: fill this in.



---

**中文对照（参考译文，不含解答）**

# 随堂问题 2

ViT 虽然在大规模图像识别任务中取得了成功，但在较小的数据集上训练时，表现往往落后于传统 CNN。这种性能差距背后的根本因素是什么？可以使用哪些技术改善 ViT 在小数据集上的表现？

**你的回答：** [请自行填写]

# Inline Question 3

How does the computational cost of the self-attention layers in a ViT change if we independently make the following changes? Please ignore the computation cost of QKV and output projection.

(i) Double the hidden dimension.
(ii) Double the height and width of the input image.
(iii) Double the patch size.
(iv) Double the number of layers.

**Your Answer**: fill this in.



---

**中文对照（参考译文，不含解答）**

# 随堂问题 3

分别独立作出以下修改时，ViT 中自注意力层的计算成本会怎样变化？请忽略 QKV 投影和输出投影的计算成本。

(i) 隐藏维度扩大为两倍。
(ii) 输入图像的高度和宽度各扩大为两倍。
(iii) 图像块的边长扩大为两倍。
(iv) 网络层数扩大为两倍。

**你的回答：** [请自行填写]